In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/whole_BMRB')
print("working folder:", os.getcwd())


# Whole-BMRB set

The filters of `rBMRB/Screening_1.ipynb` except multiple entities, solid-state NMR and denatured samples; no re-referencing.

## No polypeptide

In [ ]:
import sys
MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
target_dir_2 = os.path.join(base_path, "non_polypeptide_folders")

os.makedirs(target_dir_2, exist_ok=True)
count = 0

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        entities = (entry).sequences()
        if not entities["Polymer_type"].str.contains("polypeptide").any():
            print(f"No polypeptide in {folder_path}, moving to {target_dir_2}")
            shutil.move(folder_path, os.path.join(target_dir_2, name))
            count += 1
print(f"Total entries without polypeptides: {count}")


## No readable chemical shifts

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
wrong_cs_read = []

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        try:
            cs = get_chem_shifts(entry)
        except:
            print(f"Failed to get chemical shifts for {folder_path}")
            wrong_cs_read.append(name)
            continue
print(f"Total folders with failed chemical shift read: {len(wrong_cs_read)}")

os.makedirs(os.path.join(base_path, "no_cs_read"), exist_ok=True)
no_cs_read_path = os.path.join(base_path, "no_cs_read")
for name in wrong_cs_read:
    folder_path = os.path.join(data_path, name)
    if os.path.isdir(folder_path):
        print(f"Moving {folder_path} to {no_cs_read_path}")
        shutil.move(folder_path, os.path.join(no_cs_read_path, name))

## Missing backbone shifts

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
os.makedirs(os.path.join(base_path, "missing_backbone_folders"), exist_ok=True)
missing_backbone_path = os.path.join(base_path, "missing_backbone_folders")
missing_backbone = []

for name in os.listdir(data_path):
    if name.startswith('.'):
        continue
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        df = get_chem_shifts(ms.NMRStarEntry.from_file(str_files[0]))

        df = df.loc[df.Atom_ID.isin(['H','HA','N','CA','CB','C']) | df.Atom_ID.str.contains('^HA', na=False)]

        if df.empty:
            print(f"No backbone atoms found in {folder_path}, moving to {missing_backbone_path}")
            shutil.move(folder_path, os.path.join(missing_backbone_path, name))
            missing_backbone.append(name)
            continue
print(f"Total folders with missing backbone atoms: {len(missing_backbone)}")


## Residue and length filters

In [ ]:
import makeshift as ms
import pandas as pd
import os
import glob
import pickle

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

standard_aa = {
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL"
}
standard_aa_1letter = set("ACDEFGHIKLMNPQRSTVWY")
MAX_LEN = 500

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")

result_dict = {}
failed_entries = []   # whole entry couldn't be read at all
skipped_splits = []   # (entry_key, reason) for assignments that failed a per-entity criterion

for name in os.listdir(data_path):
    if name.startswith('.'):
        continue
    folder_path = os.path.join(data_path, name)
    if not os.path.isdir(folder_path):
        continue

    str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
    if not str_files:
        print(f"No *_3.str file found in {folder_path}, skipping.")
        failed_entries.append(name)
        continue

    try:
        entry = ms.NMRStarEntry.from_file(str_files[0])
        df = get_chem_shifts(entry)
    except Exception as e:
        print(f"Failed to read {name} ({type(e).__name__}: {e}), skipping.")
        failed_entries.append(name)
        continue

    entities = entry.sequences()
    if entities.empty:
        print(f"No entity info for {name}, skipping.")
        failed_entries.append(name)
        continue

    # [MAKESHIFT] both scoped per-entity so each split below can check only
    # the entities it actually contains, instead of the whole entry at once.
    # denatured_table = entry.is_denatured()          # columns: sample, Sample_ID, Entity_ID, denatured
    # assembly_loop = entry.data_loop("assembly", "_Entity_assembly")  # may carry Entity_ID, Physical_state

    entry_data = entry.data
    assign_index = 1
    for assign_name in entry_data['assigned_chemical_shifts'].keys():
        sub_cs_full = df[df['cs_saveframe_id'] == assign_name]
        entity_ids = sorted(sub_cs_full["Entity_ID"].dropna().unique().tolist())
        if not entity_ids:
            entry_key = f"{name}_{assign_index}"
            assign_index += 1
            print(f"{entry_key}: no Entity_ID resolvable, skipping.")
            skipped_splits.append((entry_key, "no_entity_id"))
            continue

        ent_rows = entities[entities["ID"].isin(entity_ids)]

        # 1) at least one entity in this saveframe must be polypeptide
        poly_rows = ent_rows[ent_rows["Polymer_type"].str.contains("polypeptide", na=False)]
        if poly_rows.empty:
            entry_key = f"{name}_{assign_index}"
            assign_index += 1
            print(f"{entry_key}: no polypeptide entity, skipping.")
            skipped_splits.append((entry_key, "not_polypeptide"))
            continue

        # A saveframe can carry more than one polypeptide entity (e.g. a
        # homodimer sharing one assigned_chemical_shifts list). Seq_ID is
        # entity-scoped (each chain restarts at 1), so mixing chains into one
        # split would silently clash downstream. Split one result per entity.
        poly_entity_ids = poly_rows["ID"].tolist()

        for entity_id in poly_entity_ids:
            entry_key = f"{name}_{assign_index}_{entity_id}"
            assign_index += 1

            ent_cs = sub_cs_full[sub_cs_full["Entity_ID"] == entity_id]
            seq = poly_rows.loc[poly_rows["ID"] == entity_id, "Polymer_seq_one_letter_code"].iloc[0]

            if pd.isna(seq):
                print(f"{entry_key}: entity {entity_id} has no sequence, skipping.")
                skipped_splits.append((entry_key, "no_sequence"))
                continue

            # 2) no non-standard residues among this entity's assigned shifts
            unique_residues = set(ent_cs["Comp_ID"].dropna().unique())
            invalid_residues = unique_residues - standard_aa
            if invalid_residues:
                print(f"{entry_key}: non-standard Comp_ID {invalid_residues}, skipping.")
                skipped_splits.append((entry_key, "nonstandard_comp_id"))
                continue

            # 3) no non-standard amino acid in this entity's full sequence
            if not set(seq) <= standard_aa_1letter:
                print(f"{entry_key}: non-standard aa in sequence, skipping.")
                skipped_splits.append((entry_key, "nonstandard_sequence"))
                continue

            # 4) sequence length under the cutoff
            if len(seq) >= MAX_LEN:
                print(f"{entry_key}: sequence length >= {MAX_LEN}, skipping.")
                skipped_splits.append((entry_key, "too_long"))
                continue

            result_dict[entry_key] = {
                "name": name,
                "assign_name": assign_name,
                "entity_id": entity_id,
                "sequence": seq,
                "chemical shifts": ent_cs,
            }

print(f"Skipped {len(failed_entries)} entries that could not be read: {failed_entries}")
print(f"Kept {len(result_dict)} assignment splits; skipped {len(skipped_splits)} splits that failed a criterion.")

with open(os.path.join(base_path, "bmrb_cs_data_split_filtered.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)


## Table without re-referencing

In [ ]:
import sys
MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

from makeshift.reref.lacs import reref_lacs, _BMRB_STATS

import numpy as np
import pandas as pd
import os
import json
import tqdm
import pickle

_N_STD_OUTLIER = 5

def _bmrb_outlier(row, n_std=_N_STD_OUTLIER):
    """Same BMRB-outlier rule lacs.reref_lacs() applies internally before
    fitting. Duplicated here because reref_lacs() only returns the fitted
    offsets, not a per-row mask -- needed to rebuild reref_mask below."""
    if pd.isna(row['Val']):
        return False
    stats = _BMRB_STATS.get(str(row['Comp_ID']).upper(), {}).get(str(row['Atom_ID']).upper())
    if stats is None:
        return True
    mean, std = stats
    return abs(row['Val'] - mean) > n_std * std


def apply_filter(df, n_std=_N_STD_OUTLIER):
    """
    Adapter around makeshift.reref.lacs.reref_lacs(), rebuilding the old
    ReRef_v1.reref_lacs_() interface: (df, skip, offset_collect) with Val
    corrected in place and a reref_mask column (True = not a BMRB outlier
    AND that atom's offset was successfully fit).
    """
    df = df.copy()
    df['orig'] = df['Val'].copy()

    outlier = df.apply(lambda row: _bmrb_outlier(row, n_std), axis=1)
    df['reref_mask'] = (~outlier)

    return df


base_path = os.getcwd()
atom_types = ['H', 'N', 'CA', 'CB', 'C']
result_dict = {atom: {} for atom in atom_types}
result_dict.update({atom + "_mask": {} for atom in atom_types})
result_dict["entry_ID"] = {}
result_dict["sequence"] = {}

out_of_bounds = set()
mismatch = set()
notfit_refer = set()

three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

with open(os.path.join(base_path, "bmrb_cs_data_split_filtered.pkl"), 'rb') as f:
    rawdata = pickle.load(f)

processed = 0
for idx, name in enumerate(tqdm.tqdm(list(rawdata.keys())), start=1):
    # [MIGRATED] Screening_6's split step already resolved and validated the
    # sequence for the single polypeptide entity this split belongs to
    # (rejecting non-polypeptide/non-standard/too-long/denatured cases and
    # separating co-assigned chains) -- reuse it directly instead of
    # re-parsing the .str file and grabbing entities[...].values[0], which
    # would silently pick whichever entity the depositor listed first
    # (could be a binder or the wrong chain) and throw away that filtering.
    sequence = rawdata[name]["sequence"]
    df = rawdata[name]["chemical shifts"]

    key = f"{idx}"

    df['Seq_ID'] = df['Seq_ID'].astype(int)
    df = df.sort_values(by='Seq_ID').reset_index(drop=True)

    df = df.loc[
        df.Atom_ID.isin(['H', 'HA', 'N', 'CA', 'CB', 'C']) | df.Atom_ID.str.contains('^HA', na=False)
    ]
    df['Atom_ID'] = df['Atom_ID'].replace({'HA2': 'HA', 'HA3': 'HA'})
    df['Val'] = df.groupby(['Seq_ID', 'Atom_ID'])['Val'].transform('mean')
    df = df.drop_duplicates(subset=['Seq_ID', 'Atom_ID'])

    df = df.loc[
        df.Atom_ID.isin(['H', 'N', 'CA', 'CB', 'C'])
    ]
    if not df.empty:
        df = apply_filter(df)

        result_dict["entry_ID"][key] = name
        result_dict["sequence"][key] = sequence

        for atom in atom_types:
            cs_atom = np.full(len(sequence), None, dtype=object)
            cs_atom_mask = np.full(len(sequence), False, dtype=bool)
            sub_df = df[df['Atom_ID'] == atom]
            for _, row in sub_df.iterrows():
                seq_idx_cs = int(row['Seq_ID'])
                reref_mask = row['reref_mask']
                try:
                    if three_to_one[row["Comp_ID"]] != sequence[seq_idx_cs - 1].upper():
                        print(f"Residue mismatch at Seq_ID {row['Seq_ID']} in {name}")
                        mismatch.add(name)
                    elif reref_mask:
                        cs_atom[seq_idx_cs - 1] = row['Val']
                        # cs_atom stays None / mask stays False when reref_mask is False
                        cs_atom_mask[seq_idx_cs - 1] = reref_mask
                except IndexError:
                    print(f"Seq_ID {row['Seq_ID']} out of bounds in {name}")
                    out_of_bounds.add(name)

            result_dict[atom][key] = cs_atom.tolist()
            result_dict[atom + "_mask"][key] = cs_atom_mask.tolist()

        processed += 1

with open(os.path.join(base_path, "bmrb_cs_data_withoffset_noref_whole.json"), 'w') as f:
    json.dump(result_dict, f, indent=4)

print(f"Processed {processed} entries, results saved to bmrb_cs_data_withoffset_noref.json")
print(f"There are {len(mismatch)} entries with residue mismatches:", mismatch)
print(f"There are {len(out_of_bounds)} entries with out-of-bounds Seq_ID:", out_of_bounds)
print(f"There are {len(notfit_refer)} entries with no HA after rereferencing:", notfit_refer)


In [ ]:
# Remove mismatch entries from the saved JSON
import json, os

json_path = os.path.join(base_path, "bmrb_cs_data_withoffset_noref_whole.json")
new_json_path = os.path.join(base_path, "bmrb_cs_data_withoffset_noref_whole_v1.json")
with open(json_path, 'r') as f:
    data = json.load(f)

mismatch = ['bmr52649_1_1', 'bmr51689_1_1', 'bmr19825_1_1', 'bmr50825_1_1', 'bmr50088_1_1', 'bmr50824_1_1', 'bmr52460_1_1', 'bmr52026_1_1', 'bmr50380_1_1', 'bmr18265_1_1', 'bmr50826_1_1', 'bmr26648_1_1', 'bmr27254_1_1', 'bmr26875_1_1', 'bmr27587_1_1', 'bmr50204_1_1']

# Find numeric keys whose entry_ID is in the mismatch set
mismatch_keys = {k for k, v in data["entry_ID"].items() if v in mismatch}
print(f"Removing {len(mismatch_keys)} mismatch entries: {[data['entry_ID'][k] for k in mismatch_keys]}")

# Remove those keys from every field
for field in data:
    for k in mismatch_keys:
        data[field].pop(k, None)

with open(new_json_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Done. {len(data['entry_ID'])} entries remaining in {new_json_path}")

## Structures

In [ ]:
from pathlib import Path
import shutil
import json

# rBMRB_v2_1/structures_cleaned (the AF2 models this cell copied from) is not distributed; the copied
# structures are provided under data/curation (curation/README.md), so the cell is skipped when it is missing.
SRC_DIR = Path("../rBMRB_v2_1/structures_cleaned")
if SRC_DIR.exists():
    DST_DIR = Path("structures")

    DST_DIR.mkdir(parents=True, exist_ok=True)

    three_to_one = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

    def get_pdb_chain_sequences(pdb_path):
        """One-letter sequence per chain from ATOM records, first MODEL only
        (files with no MODEL/ENDMDL at all -- like these single-conformer
        exports -- are read whole). Residues are de-duplicated by (chain,
        resSeq) so multi-atom residues collapse to a single letter."""
        seqs = {}
        seen = set()
        in_first_model = True
        with open(pdb_path) as fh:
            for line in fh:
                if line.startswith("MODEL"):
                    in_first_model = line[10:14].strip() in ("", "1")
                    continue
                if line.startswith("ENDMDL"):
                    if in_first_model:
                        break
                    continue
                if not in_first_model or not line.startswith("ATOM"):
                    continue
                res_name = line[17:20].strip()
                chain_id = line[21].strip() or " "
                res_seq = line[22:26].strip()
                key = (chain_id, res_seq)
                if key in seen:
                    continue
                seen.add(key)
                one = three_to_one.get(res_name)
                if one:
                    seqs.setdefault(chain_id, []).append(one)
        return {c: "".join(letters) for c, letters in seqs.items()}

    # Only the leading accession number matters for finding candidate structure
    # files -- the trailing "_<n>_<n>" in both the pdb filenames and our
    # entry_IDs (assign_index/entity_id) are unrelated numbering schemes, so we
    # deliberately ignore both suffixes at this stage and disambiguate by
    # sequence instead.
    src_lookup = {}
    for src_path in SRC_DIR.glob("*.pdb"):
        accession = src_path.stem.split("_")[0]
        src_lookup.setdefault(accession, []).append(src_path)

    with open("bmrb_cs_data_withoffset_noref_whole_v1.json", 'r') as f:
        data = json.load(f)

    missing, no_seq_match, ambiguous, copied = [], [], [], []

    for key, entry_id in data['entry_ID'].items():
        sequence = data['sequence'][key]
        accession = entry_id.removeprefix("bmr").split("_")[0]

        candidates = src_lookup.get(accession)
        if not candidates:
            missing.append(entry_id)
            continue

        matches = []  # (src_path, chain_id) whose chain sequence == our sequence exactly
        for cand in candidates:
            for chain_id, chain_seq in get_pdb_chain_sequences(cand).items():
                if chain_seq == sequence:
                    matches.append((cand, chain_id))

        if not matches:
            no_seq_match.append(entry_id)
            continue
        if len(matches) > 1:
            ambiguous.append((entry_id, [(p.name, c) for p, c in matches]))

        src_path = matches[0][0]  # multiple exact matches: only first copied
        dst_path = DST_DIR / f"{entry_id}.pdb"
        shutil.copy2(src_path, dst_path)
        copied.append(dst_path.name)

    print(f"Copied {len(copied)} files.")
    if missing:
        print(f"{len(missing)} entries had no candidate structure file (accession not found):")
        for m in missing:
            print(" ", m)
    if no_seq_match:
        print(f"{len(no_seq_match)} entries had candidate files but no exact sequence match:")
        for m in no_seq_match:
            print(" ", m)
    if ambiguous:
        print(f"{len(ambiguous)} entries matched more than one chain/file (only first copied):")
        for e, names in ambiguous:
            print(" ", e, names)
else:
    print("skipped: copied structures provided in data/curation (curation/README.md)")


In [ ]:
from pathlib import Path
import json

ENTRY_ID = "bmr25979_1_1"

with open("bmrb_cs_data_withoffset_noref_whole_v1.json") as f:
    data = json.load(f)

key = next(k for k, v in data['entry_ID'].items() if v == ENTRY_ID)
sequence = data['sequence'][key]

out_path = Path(f"{ENTRY_ID}.fasta")
with open(out_path, "w") as f:
    f.write(f">{ENTRY_ID}\n{sequence}\n")

print(f"Wrote {out_path} (len={len(sequence)})")

## Leakage filter against the rBMRB validation and test sets

In [ ]:
"""
Leakage filter for the whole BMRB (NOREF_whole) against the fixed rBMRB val/test (cluster_30p), using the same
two-step logic as the rBMRB split:
1. Foldseek clustering of pool ∪ val ∪ test with the same parameters as the rBMRB split
   (--alignment-type 1 --tmscore-threshold 0.3 --min-seq-id 0.3); drop pool entries that share a
   cluster with any val/test entry.
2. One exhaustive MMseqs2 all-against-all search over pool ∪ val ∪ test (one fixed database, so
   E-values are comparable); drop pool entries homologous to any val/test entry under the rBMRB criteria
   (identity >= 0.3, alignment covering >= 80% of either sequence, E <= 1e-3; read from criteria.json).
3. Drop pool entries that Foldseek cannot index (they cannot be checked for leakage).
No same-accession rule: the whole BMRB contains multi-entity entries, where one accession can hold different proteins;
same-protein entries are caught by the homology search.
"""
import json, subprocess, shutil
from pathlib import Path
import pandas as pd

FOLDSEEK = "foldseek"   # on PATH (curation/README.md)
MMSEQS   = "mmseqs"     # on PATH (curation/README.md)

WHOLE_POOL  = Path("../whole_BMRB/foldseek_pool")
RBMRB_ROOT  = Path("../rBMRB/cluster_30p")
RBMRB_SPLIT = RBMRB_ROOT / "structures_split_filtered"
OUT      = Path("../whole_BMRB/cluster_30p")
WORK     = OUT / "leak_filter"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

# Same homology criteria as the rBMRB split
crit = json.loads((RBMRB_ROOT / "split_output_filtered" / "criteria.json").read_text())["homology"]
MIN_ID, MIN_COV, MAX_EVALUE = crit["min_identity"], crit["min_coverage_either"], crit["max_evalue"]
THREADS  = "8"
MAX_SEQS = "20000"   # > total number of structures, so no hit list is truncated
FMT      = "query,target,fident,alnlen,qcov,tcov,evalue,bits"
print(f"criteria: identity >= {MIN_ID}, either-coverage >= {MIN_COV}, E <= {MAX_EVALUE}")

# whole-BMRB names: bmr30599_1_1 (foldseek_pool, bmr* only)
# rBMRB names: 30599_1      (cluster_30p filtered split)
ids = {"pool": sorted(p.stem for p in WHOLE_POOL.glob("bmr*.pdb")),
       "val":  sorted(p.stem for p in (RBMRB_SPLIT / "val").glob("*.pdb")),
       "test": sorted(p.stem for p in (RBMRB_SPLIT / "test").glob("*.pdb"))}
src = {p.stem: p for p in WHOLE_POOL.glob("bmr*.pdb")}
for s in ("val", "test"):
    src.update({p.stem: p for p in (RBMRB_SPLIT / s).glob("*.pdb")})
pool, heldout = set(ids["pool"]), set(ids["val"]) | set(ids["test"])
assert not set(ids["val"]) & set(ids["test"]) and not pool & heldout, "name collision between sets"
print({k: len(v) for k, v in ids.items()})

def run(*args):
    subprocess.run([str(a) for a in args], check=True, capture_output=True)

def strip(col):
    return col.astype(str).str.removesuffix(".pdb")

# 1. One combined structure DB (pool + val + test)
ALL = WORK / "all"
ALL.mkdir()
for e in ids["pool"] + ids["val"] + ids["test"]:
    (ALL / f"{e}.pdb").symlink_to(src[e].resolve())
run(FOLDSEEK, "createdb", ALL, WORK / "alldb", "--threads", THREADS)
run(FOLDSEEK, "convert2fasta", WORK / "alldb", WORK / "all.fa")
in_db = {l.split("\t")[1].removesuffix(".pdb") for l in open(WORK / "alldb.lookup")}
not_indexed = sorted((pool | heldout) - in_db)
assert not set(not_indexed) & heldout, f"val/test entries not indexed by foldseek: {set(not_indexed) & heldout}"
print(f"not indexed by foldseek (pool): {not_indexed}")

reasons = {}
def mark(entries, why):
    for e in entries:
        reasons.setdefault(e, []).append(why)

# 2. Foldseek clustering, same parameters as the rBMRB split
run(FOLDSEEK, "easy-cluster", ALL, WORK / "fs", WORK / "tmp_fs_cluster",
    "--alignment-type", "1", "--tmscore-threshold", "0.3", "--min-seq-id", "0.3",
    "--cov-mode", "0", "--threads", THREADS)
fs = pd.read_csv(WORK / "fs_cluster.tsv", sep="\t", names=["rep", "member"], dtype=str)
fs["rep"], fs["member"] = strip(fs["rep"]), strip(fs["member"])
held_clusters = set(fs.loc[fs["member"].isin(heldout), "rep"])
fs_drop = set(fs.loc[fs["rep"].isin(held_clusters) & fs["member"].isin(pool), "member"])
mark(fs_drop, "foldseek_cluster")
print(f"foldseek clusters: {fs['rep'].nunique()} | containing val/test: {len(held_clusters)} "
      f"| pool entries in them: {len(fs_drop)}")

# 3. One exhaustive MMseqs2 all-against-all search over the same set
m8 = WORK / "all_vs_all.m8"
run(MMSEQS, "easy-search", WORK / "all.fa", WORK / "all.fa", m8, WORK / "tmp_mm",
    "--exhaustive-search", "1", "-e", "10", "--max-seqs", MAX_SEQS, "--threads", THREADS,
    "--format-output", FMT)
aln = pd.read_csv(m8, sep="\t", names=FMT.split(","))
aln["query"], aln["target"] = strip(aln["query"]), strip(aln["target"])
hits = aln[(aln["query"] != aln["target"])
           & (aln.fident >= MIN_ID)
           & (aln[["qcov", "tcov"]].max(axis=1) >= MIN_COV)
           & (aln.evalue <= MAX_EVALUE)]
cross = hits[(hits["query"].isin(pool) & hits["target"].isin(heldout))
             | (hits["query"].isin(heldout) & hits["target"].isin(pool))]
seq_drop = set(cross["query"][cross["query"].isin(pool)]) | set(cross["target"][cross["target"].isin(pool)])
mark(seq_drop, "seq_homolog")

# 4. Not indexed by foldseek (cannot be checked for leakage)
mark(not_indexed, "not_indexed")

train_keep = [e for e in ids["pool"] if e not in reasons]

# 5. Write list, drop log and criteria
(OUT / "train_entry_ids.txt").write_text("\n".join(train_keep) + "\n")
pd.DataFrame([(e, ";".join(sorted(set(r)))) for e, r in sorted(reasons.items())],
             columns=["entry", "reason"]).to_csv(OUT / "train_dropped.tsv", sep="\t", index=False)
cross.to_csv(OUT / "pool_heldout_seq_hits.tsv", sep="\t", index=False)
reason_counts = pd.Series([r for rs in reasons.values() for r in set(rs)]).value_counts()
print(f"pool {len(ids['pool'])} -> train {len(train_keep)} (dropped {len(reasons)})")
print("drop reasons (an entry can have several):", reason_counts.to_dict())
(OUT / "criteria.json").write_text(json.dumps({
    "heldout": "rBMRB/cluster_30p val + test (fixed)",
    "foldseek_cluster": "easy-cluster --alignment-type 1 --tmscore-threshold 0.3 --min-seq-id 0.3 on pool+val+test",
    "homology": {"min_identity": MIN_ID, "min_coverage_either": MIN_COV, "max_evalue": MAX_EVALUE,
                 "search": "MMseqs2 easy-search, exhaustive, all-against-all over pool+val+test"},
    "same_bmrb_accession_dropped": False,
    "counts": {"pool": len(ids["pool"]), "train": len(train_keep), "val": len(ids["val"]), "test": len(ids["test"])},
}, indent=2))

# 6. Verification on the same clustering / search results
keep = set(train_keep)
n_fs = int((fs["rep"].isin(held_clusters) & fs["member"].isin(keep)).sum())
n_seq = int(((hits["query"].isin(keep) & hits["target"].isin(heldout))
             | (hits["query"].isin(heldout) & hits["target"].isin(keep))).sum())
print(f"train in a val/test foldseek cluster: {n_fs} | train-val/test homologous pairs: {n_seq}")
assert n_fs == 0 and n_seq == 0, "leakage remains between train and val/test"
print("OK: no foldseek-cluster or sequence-homology overlap between train and val/test")


## Training set

In [ ]:
"""
Copy the filtered train structures into a fresh directory. Always starts empty so
no stale files from an earlier run can leak into training.
"""
import shutil
from pathlib import Path

OUT = Path("../whole_BMRB/cluster_30p")
WHOLE_POOL      = Path("../whole_BMRB/foldseek_pool")
TRAIN_STRUCT = OUT / "structures_train"

train_keep = [l.strip() for l in open(OUT / "train_entry_ids.txt") if l.strip()]

if TRAIN_STRUCT.exists():
    shutil.rmtree(TRAIN_STRUCT)
TRAIN_STRUCT.mkdir(parents=True)

missing = []
for e in train_keep:
    src_path = WHOLE_POOL / f"{e}.pdb"
    if not src_path.exists():
        missing.append(e)
        continue
    shutil.copy2(src_path, TRAIN_STRUCT / f"{e}.pdb")

n_files = len(list(TRAIN_STRUCT.glob("*.pdb")))
print(f"copied {n_files} / {len(train_keep)} structures -> {TRAIN_STRUCT}")
assert not missing, f"{len(missing)} entries have no source pdb: {missing[:10]}"
assert n_files == len(train_keep)


In [ ]:
"""
Build train.json from the full NOREF_whole json using the filtered entry list (not a
directory scan), same schema as the source: field -> {key: value}. Also writes
train.list (entry IDs without the "bmr" prefix).
"""
import json
from pathlib import Path

OUT = Path("../whole_BMRB/cluster_30p")
SRC_JSON = Path("../whole_BMRB/bmrb_cs_data_withoffset_noref_whole_v1.json")

train_keep = {l.strip() for l in open(OUT / "train_entry_ids.txt") if l.strip()}

with open(SRC_JSON) as f:
    data = json.load(f)

kept_keys = {k for k, v in data["entry_ID"].items() if v in train_keep}
train_data = {field: {k: v for k, v in values.items() if k in kept_keys}
              for field, values in data.items()}

with open(OUT / "train.json", "w") as f:
    json.dump(train_data, f, indent=4)

matched = {data["entry_ID"][k] for k in kept_keys}
missing = sorted(train_keep - matched)
print(f"kept {len(kept_keys)} / {len(data['entry_ID'])} entries -> {OUT / 'train.json'}")
assert not missing, f"{len(missing)} train entries not found in {SRC_JSON.name}: {missing[:10]}"
assert len(kept_keys) == len(train_keep), "duplicate entry_ID values in the source json"

with open(OUT / "train.list", "w") as f:
    for k in sorted(kept_keys, key=int):
        f.write(train_data["entry_ID"][k].removeprefix("bmr") + "\n")
print(f"wrote {OUT / 'train.list'}")
